# Eve DPO — preference tuning on Kaggle

Runs after `eve_sft_kaggle.ipynb` and the local checkpoint pick. Settings → GPU **T4**, Internet **on**.

**Inputs:** attach `therapylink-corpus` (code), `therapylink-eve-sft` (the picked SFT adapter, pushed with
`tl-kaggle adapter <run>`) and a dataset holding `dpo_pairs.jsonl`.

Pairs are built **locally**, not here: `eve_dpo build-pairs` samples the picked checkpoint through llama-server
and pairs a sample that passes the hard rules against one that breaks them (see ml/KAGGLE.md). Both sides are
the model's own replies. The gold AnnoMI / ESConv turns used before taught brevity, and ESConv is non-commercial.

The SFT adapter is loaded twice: one copy trains, the frozen copy is the reference, so the KL anchor is the
SFT policy and not stock Gemma. `--rpo-alpha 0.5` keeps the chosen side's likelihood from collapsing, and the
run aborts if reward margins are still 0 at step 50.

Same three constraints as SFT, and DPO hits the second one harder: it runs two forward passes
(chosen, rejected) before each backward, which is exactly the case transformers 5.5.0 breaks.

In [ ]:
# transformers>=5.5.2: on 5.5.0/5.5.1 Gemma 4 E-series KV sharing is module-scoped, so a second
# forward before backward corrupts the first graph under gradient checkpointing. Kaggle ships 5.5.0.
# Pinned to the stack that trained v16-clean and DPO v16 (Kaggle kernel logs, 2026-09-25; peft from
# adapter_config.json). Unpinned installs cost the v16 DPO round 8 failed launches. unsloth 2026.9.11
# declares transformers<=5.5.0, so transformers goes in second, on its own, exactly as in v16: pip's
# resolver warning about it is expected. Move a pin only together with a smoke run.
PINS = {'unsloth': '2026.9.11', 'transformers': '5.17.0', 'trl': '0.24.0', 'peft': '0.19.1'}
%pip install -q "unsloth==2026.9.11" "trl==0.24.0" "peft==0.19.1" bitsandbytes typer rich pydantic
%pip install -q "transformers==5.17.0"
# trl 0.24 asks transformers whether optional packages are installed, and transformers 5.x now
# answers with a tuple, (False, None), which is truthy. So trl believes mergekit (and every other
# optional package) is present, and its DPO trainer dies importing it: "No module named 'mergekit'"
# (the first two v16 DPO launches). SFT never imports those paths. Patch trl's check to return a
# real bool again; the DPO import check below fails in seconds if that is not enough.
import importlib.util, os
_trl = os.path.dirname(importlib.util.find_spec('trl').origin)
_iu = os.path.join(_trl, 'import_utils.py')
_src = open(_iu, encoding='utf-8').read()
_PATCH = '''from transformers.utils.import_utils import _is_package_available as _tl_tf_is_package_available


def _is_package_available(name, return_version=False):  # _tl_bool_is_package_available
    r = _tl_tf_is_package_available(name, return_version=True)
    ok, ver = r if isinstance(r, tuple) else (bool(r), None)
    return (bool(ok), ver) if return_version else bool(ok)
'''
if '_tl_bool_is_package_available' not in _src:
    _src = _src.replace('from transformers.utils.import_utils import _is_package_available\n', _PATCH, 1)
    assert '_tl_bool_is_package_available' in _src, 'trl import_utils changed shape; patch did not apply'
    open(_iu, 'w', encoding='utf-8').write(_src)
# transformers 5 removed PreTrainedModel.warnings_issued; trl 0.24's DPOTrainer sets a key in it only
# to silence a token-estimate warning, and dies with AttributeError (the third v16 DPO launch).
_dpo = os.path.join(_trl, 'trainer', 'dpo_trainer.py')
_dsrc = open(_dpo, encoding='utf-8').read()
_OLD = 'model.warnings_issued["estimate_tokens"] = True'
if _OLD in _dsrc:
    _dsrc = _dsrc.replace(_OLD, 'getattr(model, "warnings_issued", {})["estimate_tokens"] = True')
    open(_dpo, 'w', encoding='utf-8').write(_dsrc)
assert _OLD not in open(_dpo, encoding='utf-8').read()
print('patched', _iu)

import glob, json, os, shutil, subprocess, sys

WORK = '/kaggle/working/tl'
os.makedirs(WORK, exist_ok=True)

REPO = os.environ.get('TL_REPO', '')
src = None
if REPO:
    try:
        subprocess.run(['git', 'clone', '--depth', '1', REPO, WORK + '/repo'], check=True)
        src = WORK + '/repo/ml'
    except subprocess.CalledProcessError:
        print('clone failed, using the dataset copy')
if src is None:
    hits = glob.glob('/kaggle/input/**/code/therapylink_ml', recursive=True)
    assert hits, 'No therapylink_ml found. Attach the therapylink-corpus dataset.'
    src = os.path.dirname(hits[0])

shutil.copytree(src, WORK + '/ml', dirs_exist_ok=True)
os.chdir(WORK + '/ml')
%pip install -q -e . --no-deps

os.makedirs('data/processed', exist_ok=True)
pairs = glob.glob('/kaggle/input/**/dpo_pairs.jsonl', recursive=True)
assert pairs, 'No dpo_pairs.jsonl found. Build pairs locally (eve_dpo build-pairs) and attach them as a dataset.'
shutil.copy(pairs[0], 'data/processed/dpo_pairs.jsonl')

# Kaggle flattens a dataset's single top-level folder, so the adapter files may sit at the dataset
# root rather than under adapter/. Find any directory holding a complete PEFT adapter.
adapters = [os.path.dirname(f) for f in glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
            if os.path.exists(os.path.join(os.path.dirname(f), 'adapter_model.safetensors'))]
assert adapters, 'No LoRA adapter found. Attach the therapylink-eve-sft dataset.'
assert len(adapters) == 1, f'More than one adapter attached, pick one: {adapters}'
shutil.copytree(adapters[0], 'runs/eve-sft/adapter', dirs_exist_ok=True)
print('adapter:', adapters[0])

# The adapter must be the picked SFT model, trained with the prompt the app will serve with it.
# v16 writes training_prompt.json; an older adapter (v12, still the first version of this dataset)
# has none and must not be preference-trained by accident.
manifest = os.path.join('runs/eve-sft/adapter', 'training_prompt.json')
assert os.path.exists(manifest), 'The attached adapter has no training_prompt.json: it predates v16. Attach the picked v16 adapter.'
print('adapter prompt:', json.load(open(manifest)))

# Kaggle can mount a previous dataset version for a while after an upload reports "ready".
_help = subprocess.run([sys.executable, '-m', 'therapylink_ml.train.eve_dpo', 'train', '--help'], capture_output=True, text=True).stdout
for _flag in ('--rpo-alpha', '--margin-check-steps', '--eval-frac', '--stop-train-acc'):
    assert _flag in _help, f'stale code in the dataset: eve_dpo train has no {_flag}. Wait for the new dataset version, then rerun.'
# Fail in seconds, not in the smoke run, if the DPO trainer still can't be imported.
subprocess.run([sys.executable, '-c', 'from trl import DPOTrainer, DPOConfig; print("trl DPO import OK")'], check=True)
n_pairs = sum(1 for _ in open('data/processed/dpo_pairs.jsonl', encoding='utf-8'))
print('pairs:', n_pairs)
assert n_pairs >= 100, f'only {n_pairs} pairs: too few to move the policy'
!python -c "import transformers, trl; print('transformers', transformers.__version__, '| trl', trl.__version__)"

# Fail in seconds if anything moved off the pins (a dependency pulling a newer trl, say).
from importlib.metadata import version as _version
_drift = {p: _version(p) for p in PINS if _version(p) != PINS[p]}
assert not _drift, f'library versions drifted from the pins: {_drift}'


In [ ]:
# Same environment as the real run below: one GPU (Gemma 4 E-series cannot be sharded) and the
# datasets cache off (DPO's map() otherwise hashes an Unsloth-patched closure with dill).
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['HF_DATASETS_DISABLE_CACHING'] = '1'
# Plain tracebacks: typer's boxed ones were cut off in Kaggle's log twice, losing the error itself.
os.environ['_TYPER_STANDARD_TRACEBACK'] = '1'

# run_dpo.py turns the cache off in-process, then runs eve_dpo. With TL_NO_UNSLOTH_DPO_PATCH=1 it also
# skips Unsloth's PatchDPOTrainer, which rewrites trl's DPO internals and is only a speed-up. Launches
# 4 and 5 died inside DPOTrainer.__init__ on Kaggle, while the same trl, tokenizer and pairs prepare
# fine locally without Unsloth, so the patch is the prime suspect.
RUN_DPO = '''import os, sys, runpy, functools, datasets
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
datasets.disable_caching()
if os.environ.get('TL_NO_UNSLOTH_DPO_PATCH') == '1':
    import unsloth
    unsloth.PatchDPOTrainer = lambda *a, **k: None
    print('Unsloth PatchDPOTrainer disabled', flush=True)
# The dataset's copy of eve_dpo predates two fixes, applied here after any Unsloth import (which can
# replace trl's classes with its own). See eve_dpo.TextDPOTrainer and precompute_ref_log_probs there.
import trl
_TLBase = trl.DPOTrainer
class TextDPOTrainer(_TLBase):
    # Gemma 4 counts as image-text-to-text; keep these text-only pairs on trl's text path.
    def _prepare_dataset(self, *args, **kwargs):
        self.is_vision_model = False
        return super()._prepare_dataset(*args, **kwargs)
trl.DPOTrainer = TextDPOTrainer
_TLConfig = trl.DPOConfig
class PrecomputeDPOConfig(_TLConfig):
    # Reference logps before training: a reference forward inside the step breaks gradient
    # checkpointing's recompute on Gemma 4's shared KV layers. functools.wraps keeps the signature,
    # which eve_dpo inspects for model_adapter_name, ref_adapter_name and rpo_alpha.
    @functools.wraps(_TLConfig.__init__)
    def __init__(self, *args, **kwargs):
        kwargs['precompute_ref_log_probs'] = True
        # Evaluate one pair at a time: 8 (trl's default) ran the T4 out of memory at the first eval.
        kwargs['per_device_eval_batch_size'] = 1
        super().__init__(*args, **kwargs)
trl.DPOConfig = PrecomputeDPOConfig
print('text-only DPO, reference logps precomputed', flush=True)
sys.argv = ['eve_dpo'] + sys.argv[1:]
runpy.run_module('therapylink_ml.train.eve_dpo', run_name='__main__')
'''
open('/kaggle/working/run_dpo.py', 'w').write(RUN_DPO)

# Smoke: the DPO trainer (two copies of the SFT adapter, one frozen as the reference) has never run.
# 32 pairs, 1 epoch, a few minutes, so a bug costs minutes instead of the real run. The margin check
# is pushed past the end (0 would fire on the first logged step); the real run keeps it at 50.
import itertools
with open('data/processed/dpo_pairs.jsonl', encoding='utf-8') as f, open('/kaggle/working/dpo_smoke.jsonl', 'w', encoding='utf-8') as g:
    g.writelines(itertools.islice(f, 32))

def smoke(no_patch):
    env = dict(os.environ, TL_NO_UNSLOTH_DPO_PATCH='1' if no_patch else '0')
    tag = 'nopatch' if no_patch else 'patch'
    out = f'/kaggle/working/runs/eve-dpo-smoke-{tag}'
    r = subprocess.run([sys.executable, '-u', '/kaggle/working/run_dpo.py', 'train', '--sft-adapter', 'runs/eve-sft/adapter',
                        '--pairs', '/kaggle/working/dpo_smoke.jsonl', '--epochs', '1', '--eval-frac', '0.25',
                        '--margin-check-steps', '1000000', '--no-gguf', '--out', out],
                       env=env, capture_output=True, text=True)
    log = f'/kaggle/working/dpo_smoke_{tag}.log'
    open(log, 'w', encoding='utf-8').write(r.stdout + '\n----- stderr -----\n' + r.stderr)
    ok = r.returncode == 0 and os.path.isfile(f'{out}/adapter/adapter_model.safetensors')
    print(f'smoke ({tag}):', 'OK' if ok else f'FAILED (exit {r.returncode}); full output in {log}')
    if not ok:
        print('\n'.join(r.stderr.strip().splitlines()[-40:]))
    shutil.rmtree(out, ignore_errors=True)
    return ok

# Without Unsloth's DPO patch first: with it, the smoke run went out of memory on launch 8.
DPO_NO_PATCH = None
if smoke(no_patch=True):
    DPO_NO_PATCH = True
elif smoke(no_patch=False):
    DPO_NO_PATCH = False
assert DPO_NO_PATCH is not None, 'DPO smoke failed with and without Unsloth PatchDPOTrainer; see dpo_smoke_*.log'
os.environ['TL_NO_UNSLOTH_DPO_PATCH'] = '1' if DPO_NO_PATCH else '0'
print('DPO smoke OK; Unsloth DPO patch', 'OFF' if DPO_NO_PATCH else 'ON')


In [ ]:
# One GPU: Gemma 4 E-series per-layer embeddings cannot be sharded, and Unsloth ignores device_map.
%env CUDA_VISIBLE_DEVICES=0
%env HF_DATASETS_DISABLE_CACHING=1
# v17 plan (ml/NEXT_ROUND.md): 800-1,000 pairs, 20% held out for eval accuracy, ONE epoch at batch 1 x 8
# (~100 optimizer steps, ~45 min on a T4). v16 ran 2 epochs on 158 pairs and memorised them: train
# reward accuracy 1.0, eval flat at 0.65. --stop-train-acc 1.0 stops a repeat of that and still saves.
# Watch rewards/margins rise and logps/chosen not fall by more than ~10%.
!python -u /kaggle/working/run_dpo.py train   --sft-adapter runs/eve-sft/adapter --pairs data/processed/dpo_pairs.jsonl   --beta 0.1 --lr 5e-6 --epochs 1 --stop-train-acc 1.0 --rpo-alpha 0.5   --max-length 1536 --max-prompt-length 1024 --eval-frac 0.2   --max-hours 10 --no-gguf   --out /kaggle/working/runs/eve-dpo

In [ ]:
# The same gate as SFT. subprocess with check=True fails the version; a `!command` would not.
import subprocess, sys
subprocess.run([sys.executable, '-u', '-m', 'therapylink_ml.train.eos_check', '/kaggle/working/runs/eve-dpo/adapter'], check=True)

In [ ]:
# Save Version before closing, or /kaggle/working is wiped.
!ls -la /kaggle/working/runs/eve-dpo/adapter 2>/dev/null || ls -la /kaggle/working/runs/